# Diagnóstico de calidad de los datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Rol:** analista de datos responsable de evaluar la calidad inicial de los datasets del proyecto (desarrollo individual)
**Aprendiz:** Juan Camilo Sarrazola Agudelo

> **Cómo está construido este notebook:** igual que en la EV04, las cifras de las interpretaciones no están escritas a mano.
> Se calculan con código a partir de los DataFrames cargados, de modo que el diagnóstico siempre corresponde a los datos.
> En esta actividad **no se limpia ni se modifica ningún registro**: solo se diagnostica, se interpreta y se documenta.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Nombre y propósito del proyecto
**Proyecto:** *Explorando Instacart: descubre patrones de compra.*

**Propósito:** describir los hábitos de compra de los clientes de Instacart (plataforma de compras de supermercado) a partir de
sus pedidos: cuándo compran, cada cuánto vuelven a comprar y qué productos repiten, para que el equipo de operaciones
pueda tomar decisiones con información confiable.

### 1.2 Pregunta analítica (recuperada de la EV04)
> **¿Qué hábitos de compra presentan los clientes de Instacart —cuándo compran, cada cuánto vuelven a comprar y qué productos recompran—?**

Se concreta en las tres preguntas de análisis formuladas en la EV04:
1. ¿Cada cuántos días, en promedio, vuelven a comprar los clientes? (`days_since_prior_order`)
2. ¿Qué proporción de los artículos de los pedidos son recompras? (`reordered`)
3. ¿En qué días y horas se concentran los pedidos de los clientes recurrentes? (`order_dow`, `order_hour_of_day`, `order_number`)

### 1.3 Datasets del proyecto y su relación con el problema

| N.º | Archivo | DataFrame | Qué contiene | Relación con la pregunta analítica |
|---|---|---|---|---|
| 1 | `instacart_orders.csv` | `orders` | Un registro por pedido: usuario, número de pedido, día, hora y días desde el pedido anterior. | Responde *cuándo* compran y *cada cuánto* vuelven. |
| 2 | `products.csv` | `products` | Catálogo: identificador, nombre, pasillo y departamento del producto. | Permite saber *qué* productos se compran. |
| 3 | `aisles.csv` | `aisles` | Tabla de consulta `aisle_id` → nombre del pasillo. | Agrupa productos por pasillo. |
| 4 | `departments.csv` | `departments` | Tabla de consulta `department_id` → nombre del departamento. | Agrupa productos por departamento. |
| 5 | `order_products.csv` | `order_products` | Detalle de pedidos: un registro por producto, posición en el carrito y si es recompra. | Responde *qué* se recompra y une pedidos con productos. |

### 1.4 Aspectos de calidad que se deben revisar antes de usar los datos
- **Completitud:** columnas con valores ausentes y si la ausencia tiene una explicación lógica.
- **Tipos de datos:** que cada variable tenga un tipo coherente con lo que representa (enteros, códigos, texto, binarios).
- **Unicidad:** filas completamente duplicadas y claves (`order_id`, `product_id`) repetidas.
- **Validez de rangos:** horas entre 0 y 23, días de la semana entre 0 y 6, `reordered` solo 0 o 1, etc.
- **Consistencia lógica:** que las variables no se contradigan entre sí (por ejemplo, primer pedido sin pedido anterior).
- **Integridad referencial:** que las claves de una tabla existan en la tabla con la que se relacionan.
- **Limitaciones:** información que falta para responder la pregunta analítica.

---
## Momento 2. Preparación del entorno y carga de datos

### 2.1 Librerías y funciones auxiliares

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


def md(texto):
    # Muestra un texto en formato Markdown dentro del notebook.
    display(Markdown(texto))


def fmt(n):
    # Entero con punto de miles (estilo es-CO): 478967 -> 478.967
    return f"{n:,.0f}".replace(",", ".")


def fmt2(x):
    # Decimal con coma decimal y punto de miles: 11.1017 -> 11,10
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(parte, total):
    # Porcentaje con dos decimales en formato es-CO.
    return fmt2(100 * parte / total)


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

### 2.2 Ubicación de los archivos
Los CSV no se versionan en GitHub (ver `README.md`). El notebook los busca primero en `data/raw/` (estructura de referencia de la guía)
y luego en `data/` (estructura usada desde la EV04). Si trabaja en Google Colab, agregue su ruta de Drive a `CANDIDATAS`.

In [ ]:
ARCHIVOS = {
    "orders": "instacart_orders.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
    "order_products": "order_products.csv",
}

CANDIDATAS = [Path("../data/raw"), Path("../data"), Path("data/raw"), Path("data")]


def encontrar_ruta_datos():
    # Devuelve la primera carpeta que contiene los cinco CSV del proyecto.
    for carpeta in CANDIDATAS:
        if all((carpeta / archivo).exists() for archivo in ARCHIVOS.values()):
            return carpeta
    raise FileNotFoundError(
        "No se encontraron los cinco CSV de Instacart. Cópielos en data/raw/ o data/ (ver README.md)."
    )


RUTA_DATOS = encontrar_ruta_datos()
print("Carpeta de datos:", RUTA_DATOS.as_posix())

### 2.3 Carga de cada dataset en un DataFrame independiente
Los cinco archivos son texto delimitado que usa **punto y coma (`;`)** como separador, por eso se cargan con `pd.read_csv(..., sep=";")`.

In [ ]:
orders = pd.read_csv(RUTA_DATOS / ARCHIVOS["orders"], sep=";")
products = pd.read_csv(RUTA_DATOS / ARCHIVOS["products"], sep=";")
aisles = pd.read_csv(RUTA_DATOS / ARCHIVOS["aisles"], sep=";")
departments = pd.read_csv(RUTA_DATOS / ARCHIVOS["departments"], sep=";")
order_products = pd.read_csv(RUTA_DATOS / ARCHIVOS["order_products"], sep=";")

# Diccionario para recorrer los cinco DataFrames con un solo bucle
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

### 2.4 Verificación de la carga
Se comprueba que cada archivo produjo un DataFrame con más de una columna (si el separador fuera incorrecto, todo quedaría en **una sola columna**)
y con al menos un registro.

In [ ]:
verificacion = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "archivo": ARCHIVOS[nombre],
            "tamaño archivo (MB)": round((RUTA_DATOS / ARCHIVOS[nombre]).stat().st_size / 1e6, 2),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "memoria en pandas (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
            "carga correcta": df.shape[0] > 0 and df.shape[1] > 1,
        }
        for nombre, df in dataframes.items()
    ]
)
assert verificacion["carga correcta"].all(), "Algún archivo no se cargó correctamente: revise el separador."
display(verificacion)
md(f"Los **{len(dataframes)}** datasets se cargaron correctamente y suman **{fmt(verificacion['filas'].sum())}** registros.")

---
## Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los ocho comandos de inspección que pide la actividad:

| Comando | Propósito |
|---|---|
| `head()` | Visualizar los primeros registros. |
| `tail()` | Visualizar los últimos registros. |
| `shape` | Identificar el número de filas y columnas. |
| `columns` | Consultar los nombres de las variables. |
| `dtypes` | Reconocer los tipos de datos. |
| `info()` | Examinar la estructura y los valores no nulos. |
| `describe()` | Estadísticas descriptivas de las variables numéricas. |
| `describe(include='object')` | Describir las variables de texto, cuando existan. |

> **Nota sobre pandas 3:** desde pandas 3.0 las columnas de texto se cargan con el tipo **`str`** y ya no como `object`.
> `describe(include='object')` las sigue incluyendo por compatibilidad (pandas emite un aviso de obsolescencia, que aquí se silencia).
> Por eso, en `dtypes` las columnas de texto aparecen como `str`.

In [ ]:
def tiene_texto(df):
    # True si el DataFrame tiene al menos una columna de texto (object o str).
    return any(pd.api.types.is_string_dtype(df[col]) or df[col].dtype == object for col in df.columns)


def inspeccionar(nombre, df):
    # Ejecuta los ocho comandos de inspección estructural sobre un DataFrame.
    md(f"### Inspección de `{nombre}`")

    md("**`head()`** · primeros 5 registros")
    display(df.head())

    md("**`tail()`** · últimos 5 registros")
    display(df.tail())

    md("**`shape`** · (filas, columnas)")
    print(df.shape)

    md("**`columns`** · nombres de las variables")
    print(list(df.columns))

    md("**`dtypes`** · tipo de dato de cada columna")
    print(df.dtypes)

    md("**`info()`** · estructura y valores no nulos")
    df.info()

    md("**`describe()`** · variables numéricas")
    display(df.describe())

    md("**`describe(include='object')`** · variables de texto")
    if tiene_texto(df):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            display(df.describe(include="object"))
    else:
        print(f"{nombre} no tiene columnas de texto: este resumen no aplica.")

### 3.1 `orders` (pedidos)

In [ ]:
inspeccionar("orders", orders)

In [ ]:
desc_o = orders.describe()
md(f'''
**Características principales de `orders`:**
- {fmt(orders.shape[0])} pedidos y {orders.shape[1]} columnas, todas numéricas. Los pedidos pertenecen a **{fmt(orders["user_id"].nunique())}** usuarios distintos.
- `order_hour_of_day` va de {desc_o.loc["min", "order_hour_of_day"]:.0f} a {desc_o.loc["max", "order_hour_of_day"]:.0f} y `order_dow` de {desc_o.loc["min", "order_dow"]:.0f} a {desc_o.loc["max", "order_dow"]:.0f}: códigos de hora y de día de la semana.
- `order_number` va de {desc_o.loc["min", "order_number"]:.0f} a {desc_o.loc["max", "order_number"]:.0f} (mediana {desc_o.loc["50%", "order_number"]:.0f}).
- `days_since_prior_order` es la única columna `float64`: tiene `count` = {fmt(desc_o.loc["count", "days_since_prior_order"])} frente a {fmt(len(orders))} filas, y va de {desc_o.loc["min", "days_since_prior_order"]:.0f} a {desc_o.loc["max", "days_since_prior_order"]:.0f} días.
- `orders` no tiene columnas de texto, por eso `describe(include='object')` no aplica.
''')

### 3.2 `products` (catálogo de productos)

In [ ]:
inspeccionar("products", products)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    desc_p = products.describe(include="object")
md(f'''
**Características principales de `products`:**
- {fmt(products.shape[0])} productos y {products.shape[1]} columnas: tres identificadores enteros y el nombre (`{products["product_name"].dtype}`).
- `product_name` tiene `count` = {fmt(desc_p.loc["count", "product_name"])}: faltan **{fmt(products["product_name"].isna().sum())}** nombres.
- `unique` = {fmt(desc_p.loc["unique", "product_name"])} y `freq` = {desc_p.loc["freq", "product_name"]}: ningún nombre se repite *exactamente*, aunque más adelante se revisa si se repite con otras mayúsculas.
- `aisle_id` va de {products["aisle_id"].min()} a {products["aisle_id"].max()} y `department_id` de {products["department_id"].min()} a {products["department_id"].max()}, coherente con los catálogos de pasillos y departamentos.
''')

### 3.3 `aisles` (pasillos)

In [ ]:
inspeccionar("aisles", aisles)

In [ ]:
md(f'''
**Características principales de `aisles`:**
- Tabla de consulta pequeña: {fmt(aisles.shape[0])} pasillos y {aisles.shape[1]} columnas (`aisle_id` entero y `aisle` texto).
- Los {fmt(aisles["aisle"].nunique())} nombres son únicos y no hay ausentes. Sin embargo, uno de los pasillos se llama **`{aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle"].iloc[0]}`** (id {aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle_id"].iloc[0]}): es una categoría de relleno, no un pasillo real.
''')

### 3.4 `departments` (departamentos)

In [ ]:
inspeccionar("departments", departments)

In [ ]:
md(f'''
**Características principales de `departments`:**
- {fmt(departments.shape[0])} departamentos y {departments.shape[1]} columnas, sin ausentes ni nombres repetidos.
- Igual que en `aisles`, existe un departamento llamado **`missing`** (id {departments.loc[departments["department"].eq("missing"), "department_id"].iloc[0]}) y otro llamado `other`: categorías genéricas que no describen el producto.
''')

### 3.5 `order_products` (detalle de los pedidos)

In [ ]:
inspeccionar("order_products", order_products)

In [ ]:
desc_op = order_products.describe()
md(f'''
**Características principales de `order_products`:**
- Es el dataset más grande: **{fmt(order_products.shape[0])}** registros y {order_products.shape[1]} columnas numéricas (≈{fmt(order_products.memory_usage(deep=True).sum() / 1e6)} MB en memoria).
- Cada fila es *un producto dentro de un pedido*; abarca {fmt(order_products["order_id"].nunique())} pedidos y {fmt(order_products["product_id"].nunique())} productos distintos.
- `add_to_cart_order` es `float64`, con `count` = {fmt(desc_op.loc["count", "add_to_cart_order"])} (faltan {fmt(order_products["add_to_cart_order"].isna().sum())}) y máximo {desc_op.loc["max", "add_to_cart_order"]:.0f}.
- `reordered` tiene mínimo {desc_op.loc["min", "reordered"]:.0f}, máximo {desc_op.loc["max", "reordered"]:.0f} y media {fmt2(desc_op.loc["mean", "reordered"])}: es una variable binaria; el {pct(order_products["reordered"].sum(), len(order_products))} % de los artículos son recompras.
''')

### 3.6 Tabla de diagnóstico estructural
Resumen de la inspección: filas, columnas, tipos de datos y observaciones de cada dataset. La tabla también se guarda en
`reports/diagnostico_estructural.csv` para que el informe técnico use exactamente los mismos resultados.

In [ ]:
def resumir_tipos(df):
    # Cuenta cuántas columnas hay de cada tipo, por ejemplo: 'int64 ×5, float64 ×1'.
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo} ×{n}" for tipo, n in conteo.items())


def observar(nombre, df):
    # Observaciones automáticas de la inspección estructural.
    notas = []
    nulos = df.isna().sum()
    for col, n in nulos[nulos > 0].items():
        notas.append(f"{col}: {fmt(n)} ausentes ({pct(n, len(df))} %)")
    for col in df.select_dtypes(include="float").columns:
        if (df[col].dropna() % 1 == 0).all():
            notas.append(f"{col} guarda enteros como float64")
    duplicadas = df.duplicated().sum()
    if duplicadas:
        notas.append(f"{fmt(duplicadas)} filas duplicadas")
    if not notas:
        notas.append("Sin ausentes ni filas duplicadas")
    return "; ".join(notas) + "."


diagnostico_estructural = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Archivo": ARCHIVOS[nombre],
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Tipos de datos": resumir_tipos(df),
            "Observaciones": observar(nombre, df),
        }
        for nombre, df in dataframes.items()
    ]
)

RUTA_REPORTES = RUTA_DATOS.parent / "reports" if RUTA_DATOS.name == "data" else RUTA_DATOS.parent.parent / "reports"
RUTA_REPORTES.mkdir(exist_ok=True)
diagnostico_estructural.to_csv(RUTA_REPORTES / "diagnostico_estructural.csv", index=False, encoding="utf-8")

tabla = "| Dataset | Filas | Columnas | Tipos de datos | Observaciones |\n|---|---|---|---|---|\n"
tabla += "\n".join(
    f"| `{r.Dataset}` | {fmt(r.Filas)} | {r.Columnas} | {r[4]} | {r.Observaciones} |"
    for r in diagnostico_estructural.itertuples(index=False)
)
md(tabla)

---
## Momento 4. Diagnóstico inicial de calidad
Se analizan los resultados de la inspección en cinco frentes: valores faltantes, tipos de datos, inconsistencias,
limitaciones frente a la pregunta analítica y acciones de limpieza propuestas.

### 4.1 Valores faltantes

In [ ]:
faltantes = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "columna": col,
            "registros": len(df),
            "ausentes": int(df[col].isna().sum()),
            "% ausentes": round(100 * df[col].isna().mean(), 3),
        }
        for nombre, df in dataframes.items()
        for col in df.columns
    ]
)
md("**Columnas con valores ausentes** (las demás columnas están completas):")
display(faltantes[faltantes["ausentes"] > 0].reset_index(drop=True))

Para cada columna con ausentes se busca **por qué** faltan los datos, porque la explicación define la acción de limpieza.

In [ ]:
# --- days_since_prior_order: ¿los ausentes son primeros pedidos?
nan_dias = orders[orders["days_since_prior_order"].isna()]
primeros_pedidos = (orders["order_number"] == 1).sum()
nan_son_primeros = (nan_dias["order_number"] == 1).all()

# --- product_name: ¿en qué pasillo y departamento están los productos sin nombre?
sin_nombre = products[products["product_name"].isna()]
pasillos_sin_nombre = aisles[aisles["aisle_id"].isin(sin_nombre["aisle_id"])]["aisle"].tolist()
deptos_sin_nombre = departments[departments["department_id"].isin(sin_nombre["department_id"])]["department"].tolist()
productos_en_missing = (products["aisle_id"] == 100).sum()
articulos_sin_nombre = order_products["product_id"].isin(sin_nombre["product_id"]).sum()

# --- add_to_cart_order: ¿en qué pedidos faltan las posiciones?
tam_pedido = order_products.groupby("order_id").size()
pedidos_nan_carrito = order_products.loc[order_products["add_to_cart_order"].isna(), "order_id"].unique()
tam_con_nan = tam_pedido.loc[pedidos_nan_carrito]
max_posicion = order_products["add_to_cart_order"].max()

md(f'''
**`orders.days_since_prior_order` — {fmt(len(nan_dias))} ausentes ({pct(len(nan_dias), len(orders))} %)**
- {"**Todos**" if nan_son_primeros else "No todos"} los ausentes tienen `order_number` = 1, y hay exactamente {fmt(primeros_pedidos)} primeros pedidos en la tabla.
- **Explicación:** en el primer pedido de un cliente no existe un pedido anterior, así que no hay días que contar. Es un ausente *estructural*, no un error.
- **Cómo afecta:** si se rellenara con 0 se diría que esos clientes compraron dos veces el mismo día, y el promedio de días entre compras bajaría artificialmente.

**`products.product_name` — {fmt(len(sin_nombre))} ausentes ({pct(len(sin_nombre), len(products))} %)**
- Todos los productos sin nombre están en el pasillo `{', '.join(pasillos_sin_nombre)}` y en el departamento `{', '.join(deptos_sin_nombre)}`; el pasillo 100 contiene {fmt(productos_en_missing)} productos, es decir, *solo* productos sin nombre.
- Estos productos aparecen en **{fmt(articulos_sin_nombre)}** registros de `order_products` ({pct(articulos_sin_nombre, len(order_products))} % de los artículos).
- **Cómo afecta:** se pueden contar sus compras y recompras, pero no se puede saber qué producto son ni a qué categoría pertenecen.

**`order_products.add_to_cart_order` — {fmt(order_products["add_to_cart_order"].isna().sum())} ausentes ({pct(order_products["add_to_cart_order"].isna().sum(), len(order_products))} %)**
- Los ausentes están concentrados en solo **{fmt(len(pedidos_nan_carrito))}** pedidos, y todos tienen entre {tam_con_nan.min()} y {tam_con_nan.max()} productos.
- La posición máxima registrada es {max_posicion:.0f}, aunque hay pedidos con hasta {tam_pedido.max()} productos.
- **Explicación probable:** el sistema solo guardó posiciones hasta la {max_posicion:.0f}; a partir de ahí quedaron vacías. Es una **limitación de registro** del origen.
- **Cómo afecta:** el impacto es mínimo para casi cualquier análisis, pero no se puede estudiar el orden de los productos en los carritos muy grandes.
''')

### 4.2 Tipos de datos
Se compara el tipo que asignó pandas con lo que cada variable representa realmente.

In [ ]:
# Qué representa cada columna (definido a partir del Momento 1)
significado = {
    ("orders", "order_id"): "identificador",
    ("orders", "user_id"): "identificador",
    ("orders", "order_number"): "conteo",
    ("orders", "order_dow"): "código de día",
    ("orders", "order_hour_of_day"): "hora",
    ("orders", "days_since_prior_order"): "conteo de días",
    ("products", "product_id"): "identificador",
    ("products", "product_name"): "texto libre",
    ("products", "aisle_id"): "identificador",
    ("products", "department_id"): "identificador",
    ("aisles", "aisle_id"): "identificador",
    ("aisles", "aisle"): "categoría",
    ("departments", "department_id"): "identificador",
    ("departments", "department"): "categoría",
    ("order_products", "order_id"): "identificador",
    ("order_products", "product_id"): "identificador",
    ("order_products", "add_to_cart_order"): "posición",
    ("order_products", "reordered"): "binaria (0/1)",
}


def evaluar_tipo(nombre, col, representa):
    s = dataframes[nombre][col]
    dtype = str(s.dtype)
    if representa in ("identificador", "conteo", "posición", "conteo de días", "hora"):
        if pd.api.types.is_integer_dtype(s):
            return "Sí", "Entero, coherente con lo que representa."
        solo_enteros = (s.dropna() % 1 == 0).all()
        return ("Parcial",
                f"Representa enteros{' y todos sus valores son enteros' if solo_enteros else ''}, pero es {dtype} "
                f"porque tiene {fmt(s.isna().sum())} NaN. Conviene `Int64` (entero que admite nulos).")
    if representa == "código de día":
        return ("Parcial", f"Entero 0–6 sin diccionario de días: es una categoría ordenada, no una cantidad (su promedio no tiene sentido).")
    if representa == "binaria (0/1)":
        valores = sorted(s.unique().tolist())
        return ("Parcial", f"Solo contiene {valores}: se puede convertir a `bool` para dejar explícito que es sí/no.")
    if representa == "categoría":
        return ("Sí", f"Texto ({dtype}) con {s.nunique()} valores únicos; podría usarse `category` para ahorrar memoria.")
    return ("Sí", f"Texto ({dtype}), coherente con un nombre de producto.")


filas_tipos = []
for (nombre, col), representa in significado.items():
    coherente, comentario = evaluar_tipo(nombre, col, representa)
    filas_tipos.append({"DataFrame": nombre, "columna": col, "dtype": str(dataframes[nombre][col].dtype),
                        "representa": representa, "¿coherente?": coherente, "comentario": comentario})
tipos = pd.DataFrame(filas_tipos)
assert len(tipos) == sum(df.shape[1] for df in dataframes.values()), "Falta evaluar alguna columna"
display(tipos)

In [ ]:
parciales = tipos[tipos["¿coherente?"] != "Sí"]
md(f'''
**Interpretación de los tipos de datos:** de {len(tipos)} columnas, {fmt((tipos["¿coherente?"] == "Sí").sum())} tienen un tipo coherente y
{len(parciales)} son *parcialmente* coherentes: {", ".join(f"`{r.DataFrame}.{r.columna}`" for r in parciales.itertuples(index=False))}.
- Las columnas `float64` en realidad guardan **enteros**: pandas las convierte a decimal solo porque `NaN` es un valor decimal.
- `order_dow` es un **código**: hacer cálculos como su promedio no tiene interpretación; además, el archivo no dice qué día es el 0.
- Ninguna columna numérica quedó como texto, lo que confirma que el separador `;` se aplicó bien y que no hay símbolos extraños en los números.
''')

### 4.3 Posibles inconsistencias

#### a) Filas duplicadas y claves repetidas

In [ ]:
claves = {
    "orders": ["order_id"],
    "products": ["product_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
    "order_products": ["order_id", "product_id"],
}
duplicados = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "filas duplicadas completas": int(df.duplicated().sum()),
            "clave": " + ".join(claves[nombre]),
            "claves repetidas": int(df.duplicated(subset=claves[nombre]).sum()),
        }
        for nombre, df in dataframes.items()
    ]
)
display(duplicados)

filas_dup = orders[orders.duplicated(keep=False)].sort_values("order_id")
md(f"**Pedidos duplicados en `orders`** ({fmt(orders.duplicated().sum())} copias sobrantes de {fmt(filas_dup['order_id'].nunique())} pedidos):")
display(filas_dup.head(10))

dias_dup = filas_dup["order_dow"].unique().tolist()
horas_dup = filas_dup["order_hour_of_day"].unique().tolist()
dia_dup = ", ".join(str(v) for v in dias_dup)
hora_dup = ", ".join(str(v) for v in horas_dup)
mismo_momento = orders[(orders["order_dow"].isin(dias_dup)) & (orders["order_hour_of_day"].isin(horas_dup))]
md(f'''
- Los duplicados son filas **idénticas en las {orders.shape[1]} columnas**, no pedidos distintos con el mismo número.
- Todos ocurren en el mismo momento: `order_dow` = {dia_dup} y `order_hour_of_day` = {hora_dup}. En esa franja hay {fmt(len(mismo_momento))} pedidos,
  de los cuales {fmt(mismo_momento.duplicated().sum())} son copias. Esa concentración sugiere un **fallo puntual del sistema** al registrar pedidos, no compras repetidas.
- **Impacto:** cuentan dos veces el mismo pedido e inflan los conteos de esa franja horaria.
''')

#### b) Validez de rangos y coherencia lógica entre variables

In [ ]:
reglas = [
    ("orders", "order_hour_of_day entre 0 y 23", ~orders["order_hour_of_day"].between(0, 23)),
    ("orders", "order_dow entre 0 y 6", ~orders["order_dow"].between(0, 6)),
    ("orders", "order_number mayor o igual a 1", orders["order_number"] < 1),
    ("orders", "days_since_prior_order entre 0 y 30 (sin contar NaN)",
     orders["days_since_prior_order"].notna() & ~orders["days_since_prior_order"].between(0, 30)),
    ("orders", "si order_number = 1, days_since_prior_order es NaN",
     (orders["order_number"] == 1) & orders["days_since_prior_order"].notna()),
    ("orders", "si days_since_prior_order es NaN, order_number = 1",
     orders["days_since_prior_order"].isna() & (orders["order_number"] != 1)),
    ("order_products", "reordered solo 0 o 1", ~order_products["reordered"].isin([0, 1])),
    ("order_products", "add_to_cart_order mayor o igual a 1 (sin contar NaN)", order_products["add_to_cart_order"] < 1),
]
validacion = pd.DataFrame(
    [{"DataFrame": df, "regla": regla, "registros que la incumplen": int(mascara.sum())} for df, regla, mascara in reglas]
)

# Regla entre tablas: un artículo no puede ser recompra en el primer pedido del cliente
primer_pedido = order_products.merge(orders.drop_duplicates()[["order_id", "order_number"]], on="order_id", how="inner")
recompra_en_primero = int(((primer_pedido["order_number"] == 1) & (primer_pedido["reordered"] == 1)).sum())
validacion.loc[len(validacion)] = ["order_products + orders", "si order_number = 1, reordered = 0", recompra_en_primero]
display(validacion)

md(f'''
**{"Ninguna regla se incumple" if validacion["registros que la incumplen"].sum() == 0 else "Hay reglas incumplidas"}:** los rangos de horas, días y códigos son válidos
y las variables son coherentes entre sí (ausencia de días ⇔ primer pedido; no hay recompras en el primer pedido).
''')

#### c) Valores sospechosos en las distribuciones

In [ ]:
dias = orders["days_since_prior_order"].value_counts().sort_index()
n_30, n_29 = dias.loc[30.0], dias.loc[29.0]
n_0 = dias.loc[0.0]
max_order_number = orders["order_number"].max()
n_max_order = (orders["order_number"] == max_order_number).sum()

md("**Frecuencia de los últimos valores de `days_since_prior_order`:**")
display(dias.tail(5).to_frame("pedidos"))
md(f'''
- El valor **30** aparece {fmt(n_30)} veces ({pct(n_30, dias.sum())} % de los valores disponibles), frente a {fmt(n_29)} del valor 29.
  Un salto así indica que 30 es un **tope**: probablemente significa «30 días o más», no exactamente 30. Promediar la columna subestima la espera real de esos clientes.
- El valor **0** aparece {fmt(n_0)} veces: clientes con dos pedidos el mismo día. Es posible (un pedido olvidado), pero conviene tenerlo en cuenta.
- `order_number` llega como máximo a {max_order_number} ({fmt(n_max_order)} pedidos con ese valor): podría ser también un límite del historial guardado.
''')

#### d) Inconsistencias en el texto de los nombres de productos

In [ ]:
nombres = products["product_name"].dropna()
nombre_normalizado = nombres.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
repetidos_mayus = nombre_normalizado.duplicated().sum()
todo_mayus = (nombres == nombres.str.upper()).sum()
espacios_extremos = (nombres != nombres.str.strip()).sum()
espacios_dobles = nombres.str.contains("  ", regex=False).sum()

ejemplo = products.loc[nombres[nombre_normalizado.duplicated(keep=False)].index]
ejemplo = ejemplo.assign(_n=nombre_normalizado).sort_values("_n").drop(columns="_n")
md("**Ejemplos de productos cuyo nombre se repite al ignorar mayúsculas y espacios:**")
display(ejemplo.head(6))

md(f'''
- **{fmt(repetidos_mayus)}** nombres se repiten si se ignoran mayúsculas y espacios (por ejemplo, el mismo producto escrito en MAYÚSCULAS y en tipo título),
  con identificadores distintos. Pueden ser el mismo producto registrado dos veces o variantes reales: requiere revisión.
- {fmt(todo_mayus)} nombres están completamente en mayúsculas, {fmt(espacios_dobles)} tienen espacios dobles y {fmt(espacios_extremos)} tienen espacios al inicio o al final.
- **Impacto:** al contar «productos más comprados» por nombre, las ventas de un mismo producto quedarían repartidas en dos filas.
''')

#### e) Integridad referencial entre tablas

In [ ]:
pedidos_sin_detalle = ~orders["order_id"].isin(order_products["order_id"])
integridad = pd.DataFrame(
    [
        ("order_products.order_id existe en orders", int((~order_products["order_id"].isin(orders["order_id"])).sum())),
        ("order_products.product_id existe en products", int((~order_products["product_id"].isin(products["product_id"])).sum())),
        ("products.aisle_id existe en aisles", int((~products["aisle_id"].isin(aisles["aisle_id"])).sum())),
        ("products.department_id existe en departments", int((~products["department_id"].isin(departments["department_id"])).sum())),
        ("orders.order_id tiene detalle en order_products", int(pedidos_sin_detalle.sum())),
    ],
    columns=["relación comprobada", "registros sin correspondencia"],
)
display(integridad)

sin_detalle = orders[pedidos_sin_detalle]
md(f'''
- Todas las claves foráneas apuntan a registros existentes: no hay productos, pasillos ni departamentos «huérfanos».
- Sin embargo, **{fmt(len(sin_detalle))}** pedidos de `orders` ({pct(len(sin_detalle), len(orders))} %) **no tienen productos** en `order_products`.
  Ninguno es un primer pedido (su `order_number` mínimo es {sin_detalle["order_number"].min()}), así que no es un patrón de clientes nuevos:
  `order_products` parece ser una **muestra parcial** del detalle de los pedidos.
- **Impacto:** al unir `orders` con `order_products` esos pedidos desaparecen (unión interna) o quedan sin productos (unión externa); hay que decidirlo de forma explícita.
''')